# Test cases requiring or benefiting from the context of a notebook

If the notebook runs successfully from start to finish, the test is successful!

TODO(all): Add additional tests and/or tests with particular assertions, as we encounter Python package version incompatibilities not currently detected by these tests.

In general, only add test cases here that require the context of a notebook. This is because this notebook, as currently written, will abort at the **first** failure. Compare this to a proper test suite where all cases are run, giving much more information about the full extent of any problems encountered.

In [ ]:
import pandas_gbq


# Package versions

In [ ]:
!pip3 freeze

In [ ]:
!pip3 show plotnine pandas google-cloud-storage google-cloud-bigquery google-resumable-media

# Test cases requiring the context of a notebook 

# Test cases benefiting from the context of a notebook 

In [ ]:
! java -version

In [ ]:
! gatk --version

In [ ]:
! samtools --version

In [ ]:
! bcftools --version

In [ ]:
! bedtools --version

## GATK python env

gcnvkernel asserts the pymc version from GATK's gatkcondaenv.yml (5.10.1) on import.

In [ ]:
%%bash
set -euo pipefail
$GATK_CONDA_ENV_HOME/bin/python -c "import gcnvkernel, scorevariants, gatktool, pymc; print('pymc', pymc.__version__)"

## nbconvert loads in the notebook env

jupyter-contrib-nbextensions broke `import nbconvert` under Notebook 7 (it imports `notebook.services`).

In [ ]:
import nbconvert.nbconvertapp
from nbconvert.exporters import get_export_names
print(get_export_names())

## Test Run the default 1D model on the VCF with NVScoreVariants

GATK 4.6.1.0 removed CNNScoreVariants; NVScoreVariants (pytorch) replaces it. It runs the model in python, so its inputs
must be local files. The b37 chr20 reference and VCF come from the public `gatk-tutorials` bucket.

In [ ]:
%%bash
set -euo pipefail
BUCKET=https://storage.googleapis.com/gatk-tutorials/workshop_2002/2-germline
mkdir -p nvscore && cd nvscore
for f in ref/ref.fasta ref/ref.fasta.fai ref/ref.dict CNNScoreVariants/vcfs/g94982_b37_chr20_1m_895.vcf; do
    curl -sSfL -o "$(basename $f)" "$BUCKET/$f"
done
gatk NVScoreVariants \
    -V g94982_b37_chr20_1m_895.vcf \
    -R ref.fasta \
    -O output_my_1d_cnn_scored.vcf
grep -c CNN_1D output_my_1d_cnn_scored.vcf